[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C68_Eval_Infrastructure_Course/05_online/05_online_monitoring.ipynb)

# 05 · 线上监控与漂移（离线-在线相关性 / PSI 与 KS / guardrail / 抽样 / 无标签信号 / 闭环）

目标：把「离线涨了线上没涨」从一个玄学问题，变成**可以按顺序诊断的三步**。

本 notebook 你会亲手实现：
1. **离线-在线的两层不匹配** —— 分布偏移 vs 口径不同，两者的诊断与修法完全不同
2. **PSI 与 KS** —— 以及「PSI 高不等于变差了」这个必须说清的局限
3. **无标签信号** —— 自一致性 / 行为分布 / 重问率，三者覆盖三个失败方向
4. **guardrail 分层** —— 误伤率与触发率的权衡，以及为什么 L2 不该直接拒答
5. **抽样策略** —— 均匀 vs 分层 vs 异常驱动；尾部采样的选择偏倚
6. **闭环健康度** —— 回灌延迟、复发率、任务集来源构成的上限

> 心智模型：**无标签监控能告诉你「输入/输出变了」，不能告诉你「变差了」。
> 漂移告警的正确动作是触发一次带标签的抽样评测，而不是直接回滚。**

## 0 · 环境与一个可控的「线上系统」

In [ ]:
import os, json, math, random, itertools
from collections import Counter, defaultdict

import numpy as np

def make_traffic(n, rng, intent_mix=None, len_scale=1.0, shift=0.0):
    """生成线上请求。intent 是场景类别，latent 是「这条请求有多难」。"""
    mix = intent_mix or {'faq': 0.55, 'billing': 0.25, 'troubleshoot': 0.15, 'other': 0.05}
    intents = list(mix); probs = np.array([mix[i] for i in intents], dtype=float)
    probs = probs / probs.sum()
    idx = rng.choice(len(intents), size=n, p=probs)
    latent = np.clip(rng.beta(2, 2, n) + shift, 0, 1)
    return [{'req_id': f'r{i:06d}', 'intent': intents[k],
             'length': int(np.clip(rng.lognormal(4.2, 0.6) * len_scale, 5, 4000)),
             'latent': float(l)}
            for i, (k, l) in enumerate(zip(idx, latent))]

BASE_QUALITY = {'faq': 0.92, 'billing': 0.70, 'troubleshoot': 0.45, 'other': 0.55}

def serve(reqs, rng, model_shift=0.0, verbosity=1.0):
    """模拟线上服务：产出回答的质量（我们知道，系统不知道）与可观测的行为指标。"""
    out = []
    for r in reqs:
        p = float(np.clip(BASE_QUALITY[r['intent']] + model_shift - 0.5 * (r['latent'] - 0.5),
                          0.01, 0.99))
        good = bool(rng.random() < p)
        out.append({**r, 'good': good,
                    'resp_len': int(np.clip(rng.lognormal(4.6, 0.5) * verbosity, 10, 6000)),
                    'md_density': float(np.clip(rng.beta(2, 6) * verbosity, 0, 1)),
                    'refused': bool(rng.random() < 0.02),
                    # 重问率：回答不好时用户更可能换个说法再问一遍
                    'reasked': bool(rng.random() < (0.42 if not good else 0.06))})
    return out

rng = np.random.default_rng(0)
online = serve(make_traffic(20000, rng), rng)
print(f'线上样本 {len(online)} 条')
print('意图分布:', {k: round(v / len(online), 3)
                    for k, v in Counter(o['intent'] for o in online).most_common()})
print(f"真实质量（系统看不到）: {np.mean([o['good'] for o in online]):.1%}")
print(f"重问率（系统看得到）:   {np.mean([o['reasked'] for o in online]):.1%}")
print('\n✅ 「真实质量」我们知道但系统不知道——这让我们能验证「无标签信号有没有用」。')

## 1 · 离线-在线的两层不匹配

离线指标估计的是 $\mathbb{E}_{x\sim D_{off}}[s(x)]$，你关心的是 $\mathbb{E}_{x\sim D_{on}}[u(x)]$。
中间隔着**分布**与**口径**两层，两层的修法完全不同。

In [ ]:
# 离线任务集：人工构造，偏向 troubleshoot（因为"难题才值得写成题"）
OFFLINE_MIX = {'faq': 0.15, 'billing': 0.25, 'troubleshoot': 0.55, 'other': 0.05}
offline = serve(make_traffic(2000, np.random.default_rng(1), intent_mix=OFFLINE_MIX),
                np.random.default_rng(1))

def coverage_report(offline, online):
    off_mix = Counter(o['intent'] for o in offline)
    on_mix = Counter(o['intent'] for o in online)
    n_off, n_on = len(offline), len(online)
    rows = []
    for k in sorted(set(off_mix) | set(on_mix)):
        rows.append((k, off_mix[k] / n_off, on_mix[k] / n_on))
    return rows

print(f"{'意图':<14}{'离线占比':>10}{'线上占比':>10}{'比值':>8}")
for k, po, pn in coverage_report(offline, online):
    print(f'{k:<14}{po:>10.1%}{pn:>10.1%}{(po/pn if pn else float("inf")):>8.2f}')

off_score = np.mean([o['good'] for o in offline])
on_score = np.mean([o['good'] for o in online])
print(f'\n离线分数 {off_score:.1%} | 线上真实质量 {on_score:.1%} | 差 {off_score-on_score:+.1%}')
assert abs(off_score - on_score) > 0.08, '分布不同 → 两个数字本来就不该相等'
print('✅ 第一层不匹配（分布）：离线任务集里 55% 是 troubleshoot，线上只有 15%。')
print('   → 离线分数系统性偏低，而且**离线的提升主要发生在线上不重要的场景上**。')

In [ ]:
# 用线上分布对离线分数做重加权 —— 这才是可比的口径（= 模块 03 的直接标准化）
def reweight_to_online(offline, online):
    on_mix = Counter(o['intent'] for o in online)
    n_on = len(online)
    by_intent = defaultdict(list)
    for o in offline:
        by_intent[o['intent']].append(float(o['good']))
    num = den = 0.0
    for k, vals in by_intent.items():
        w = on_mix[k] / n_on
        num += w * float(np.mean(vals))
        den += w
    return num / den if den else float('nan')

rw = reweight_to_online(offline, online)
print(f'离线原始   {off_score:.1%}')
print(f'离线重加权 {rw:.1%}   ← 用线上意图分布加权')
print(f'线上真实   {on_score:.1%}')
assert abs(rw - on_score) < abs(off_score - on_score), '重加权后应当更接近线上'
print(f'\n✅ 重加权把差距从 {abs(off_score-on_score):.1%} 缩到 {abs(rw-on_score):.1%}——')
print('   剩下的差距才是「口径不同」等其他原因。')
print('   → **诊断顺序：先查覆盖率（分布），再谈效应量。**')
print('   很多团队直接从「是不是提升太小了」开始，于是永远在讨论要不要做更大的改动。')

In [ ]:
# 第二层不匹配（口径）：离线测"答案对不对"，线上看"用户满不满意"
def corr(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ac, bc = a - a.mean(), b - b.mean()
    d = math.sqrt(float((ac**2).sum()) * float((bc**2).sum()))
    return float((ac*bc).sum()/d) if d else float('nan')

good = np.array([o['good'] for o in online], float)
satisfied = np.array([not o['reasked'] for o in online], float)
print(f'「答案对不对」与「用户没重问」的相关: {corr(good, satisfied):.3f}')
assert 0.2 < corr(good, satisfied) < 0.9
print('✅ 相关但远非等同——这就是第二层不匹配。')
print('   报告规范：**如果离线指标只是代理指标，必须显式声明**，')
print('   并给出它与线上指标在同一批样本上的相关系数。')

## 2 · 漂移检测：PSI 与 KS，以及它们的盲区

In [ ]:
def psi(expected, actual, bins=10, eps=1e-6):
    """Population Stability Index。分箱边界由基线（expected）决定 —— 这一点很重要。"""
    e = np.asarray(expected, float); a = np.asarray(actual, float)
    edges = np.quantile(e, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    pe, _ = np.histogram(e, bins=edges); pa, _ = np.histogram(a, bins=edges)
    pe = pe / max(pe.sum(), 1) + eps
    pa = pa / max(pa.sum(), 1) + eps
    return float(np.sum((pa - pe) * np.log(pa / pe)))

def ks_stat(a, b):
    a = np.sort(np.asarray(a, float)); b = np.sort(np.asarray(b, float))
    allv = np.concatenate([a, b])
    fa = np.searchsorted(a, allv, 'right') / len(a)
    fb = np.searchsorted(b, allv, 'right') / len(b)
    return float(np.max(np.abs(fa - fb)))

def psi_categorical(expected, actual, eps=1e-6):
    ce, ca = Counter(expected), Counter(actual)
    keys = set(ce) | set(ca)
    ne, na = max(len(expected), 1), max(len(actual), 1)
    tot = 0.0
    for k in keys:
        pe = ce[k]/ne + eps; pa = ca[k]/na + eps
        tot += (pa - pe) * math.log(pa / pe)
    return float(tot)

base = online
SCENARIOS = {
    '无漂移':        serve(make_traffic(20000, np.random.default_rng(10)), np.random.default_rng(10)),
    '请求变长':      serve(make_traffic(20000, np.random.default_rng(11), len_scale=1.8),
                          np.random.default_rng(11)),
    '意图构成变了':  serve(make_traffic(20000, np.random.default_rng(12),
                                        intent_mix={'faq':0.30,'billing':0.25,
                                                    'troubleshoot':0.40,'other':0.05}),
                          np.random.default_rng(12)),
    '模型变啰嗦':    serve(make_traffic(20000, np.random.default_rng(13)),
                          np.random.default_rng(13), verbosity=1.9),
}
print(f"{'场景':<16}{'PSI(请求长度)':>16}{'PSI(意图)':>12}{'KS(响应长度)':>15}{'真实质量变化':>14}")
for name, cur in SCENARIOS.items():
    p_len = psi([o['length'] for o in base], [o['length'] for o in cur])
    p_int = psi_categorical([o['intent'] for o in base], [o['intent'] for o in cur])
    k_out = ks_stat([o['resp_len'] for o in base], [o['resp_len'] for o in cur])
    dq = np.mean([o['good'] for o in cur]) - np.mean([o['good'] for o in base])
    print(f'{name:<16}{p_len:>16.4f}{p_int:>12.4f}{k_out:>15.4f}{dq:>+14.1%}')

p_none = psi([o['length'] for o in base], [o['length'] for o in SCENARIOS['无漂移']])
p_long = psi([o['length'] for o in base], [o['length'] for o in SCENARIOS['请求变长']])
assert p_none < 0.1 and p_long > 0.25, 'PSI 应当只在真漂移时报警'
dq_verbose = np.mean([o['good'] for o in SCENARIOS['模型变啰嗦']]) - np.mean([o['good'] for o in base])
k_verbose = ks_stat([o['resp_len'] for o in base],
                    [o['resp_len'] for o in SCENARIOS['模型变啰嗦']])
assert k_verbose > 0.3 and abs(dq_verbose) < 0.03
print('\n⚠️ 看最后一行：模型变啰嗦了（KS=0.4+ 强烈报警），但**真实质量几乎没变**。')
print('✅ 这就是无标签监控的结构性局限：**它能告诉你「变了」，不能告诉你「变差了」。**')
print('   → 漂移告警的正确动作是「触发一次带标签的抽样评测」，而不是直接回滚。')

## 3 · 无标签信号：三个覆盖不同失败方向的指标

In [ ]:
def self_consistency(reqs, rng, model_shift=0.0, n_pairs=None):
    """同一请求采样两次，看两个回答是否一致（好/坏是否相同）。"""
    sel = reqs if n_pairs is None else reqs[:n_pairs]
    a = serve(sel, np.random.default_rng(101), model_shift=model_shift)
    b = serve(sel, np.random.default_rng(202), model_shift=model_shift)
    return float(np.mean([x['good'] == y['good'] for x, y in zip(a, b)]))

def behavior_profile(rows):
    return {'resp_len_p50': float(np.median([r['resp_len'] for r in rows])),
            'md_density': float(np.mean([r['md_density'] for r in rows])),
            'refusal_rate': float(np.mean([r['refused'] for r in rows])),
            'reask_rate': float(np.mean([r['reasked'] for r in rows]))}

CASES = {
    '基线':          dict(shift=0.0, verbosity=1.0),
    '模型退化 -12pp': dict(shift=-0.12, verbosity=1.0),
    '模型变啰嗦':     dict(shift=0.0, verbosity=1.9),
}
print(f"{'场景':<16}{'真实质量':>10}{'自一致性':>10}{'重问率':>10}{'响应长度P50':>14}{'md密度':>10}")
profiles = {}
for name, cfg in CASES.items():
    reqs = make_traffic(8000, np.random.default_rng(21))
    rows = serve(reqs, np.random.default_rng(22), model_shift=cfg['shift'],
                 verbosity=cfg['verbosity'])
    prof = behavior_profile(rows)
    sc = self_consistency(reqs[:3000], None, model_shift=cfg['shift'])
    profiles[name] = (np.mean([r['good'] for r in rows]), sc, prof)
    print(f'{name:<16}{profiles[name][0]:>10.1%}{sc:>10.1%}{prof["reask_rate"]:>10.1%}'
          f'{prof["resp_len_p50"]:>14.0f}{prof["md_density"]:>10.3f}')

q_base, sc_base, pr_base = profiles['基线']
q_deg, sc_deg, pr_deg = profiles['模型退化 -12pp']
q_verb, sc_verb, pr_verb = profiles['模型变啰嗦']
assert pr_deg['reask_rate'] > pr_base['reask_rate'] + 0.02, '质量退化时重问率必须上升'
assert pr_verb['resp_len_p50'] > 1.5 * pr_base['resp_len_p50'], '变啰嗦时长度必须涨'
assert abs(q_verb - q_base) < 0.03, '变啰嗦但质量没变'
print('\n✅ 三个信号覆盖三个不同的失败方向：')
print(f'   · 重问率     抓「用户没被满足」 —— 质量掉 12pp 时它涨了 '
      f'{pr_deg["reask_rate"]-pr_base["reask_rate"]:.1%}')
print(f'   · 行为分布   抓「模型行为漂移」 —— 变啰嗦时长度涨了 '
      f'{pr_verb["resp_len_p50"]/pr_base["resp_len_p50"]:.1f} 倍（而质量没变）')
print('   · 自一致性   抓「模型不稳定」')
print('   三者都不需要标注、全自动、成本近乎为零。')

In [ ]:
# 重问率作为质量的代理：适合相对比较，不适合绝对判断
qs, rs = [], []
for shift in [-0.20, -0.12, -0.06, 0.0, 0.06]:
    rows = serve(make_traffic(6000, np.random.default_rng(31)),
                 np.random.default_rng(32), model_shift=shift)
    qs.append(float(np.mean([r['good'] for r in rows])))
    rs.append(float(np.mean([r['reasked'] for r in rows])))
print(f"{'质量':>10}{'重问率':>10}")
for q, r in zip(qs, rs):
    print(f'{q:>10.1%}{r:>10.1%}')
c = corr(qs, rs)
print(f'\n质量与重问率的相关: {c:.3f}')
assert c < -0.9, '质量越高重问率越低，应当强负相关'
print('✅ 强负相关 → 重问率可以做质量的代理指标。')
print('⚠️ 但它混杂了 UI、用户习惯、流量构成——')
print('   **适合做相对比较（新旧版本同期同流量），不适合做绝对判断**')
print('   （「重问率 12% 说明质量不好」这句话没有依据）。')

## 4 · Guardrail：分层与误伤率

In [ ]:
def l0_rules(row):
    """L0 确定性规则：格式/长度/必填。零误伤（对合法输出永远不触发）。"""
    return row['resp_len'] > 5000 or row['resp_len'] < 15

def l1_small_model(row, rng, recall=0.55, fpr=0.02):
    """L1 小模型：对「坏回答」有一定召回，对好回答有小的误伤率。"""
    return (rng.random() < recall) if not row['good'] else (rng.random() < fpr)

def l2_llm_judge(row, rng, recall=0.80, fpr=0.08):
    """L2 LLM judge：召回更高，但误伤率也更高（C67：judge 的 alpha/beta）。"""
    return (rng.random() < recall) if not row['good'] else (rng.random() < fpr)

def eval_guardrail(rows, layers, seed=0):
    rng = random.Random(seed)
    trig = {name: 0 for name, _ in layers}
    fired_bad = fired_good = 0
    for r in rows:
        hit = None
        for name, fn in layers:
            if fn(r, rng) if fn.__code__.co_argcount > 1 else fn(r):
                hit = name; break
        if hit:
            trig[hit] += 1
            if r['good']:
                fired_good += 1        # 误伤
            else:
                fired_bad += 1         # 正确拦截
    n_bad = sum(1 for r in rows if not r['good'])
    n_good = len(rows) - n_bad
    return {'trigger_rate': (fired_bad + fired_good) / len(rows),
            'recall': fired_bad / n_bad if n_bad else float('nan'),
            'false_hit_rate': fired_good / n_good if n_good else float('nan'),
            'precision': fired_bad / (fired_bad + fired_good) if (fired_bad + fired_good) else float('nan'),
            'by_layer': trig}

sample = serve(make_traffic(20000, np.random.default_rng(41)), np.random.default_rng(42))
CONFIGS = {
    'L0 only':        [('L0', l0_rules)],
    'L0+L1':          [('L0', l0_rules), ('L1', l1_small_model)],
    'L0+L1+L2':       [('L0', l0_rules), ('L1', l1_small_model), ('L2', l2_llm_judge)],
}
print(f"{'配置':<14}{'触发率':>10}{'召回':>10}{'误伤率':>10}{'精确率':>10}")
res = {}
for name, layers in CONFIGS.items():
    r = eval_guardrail(sample, layers, seed=7)
    res[name] = r
    print(f'{name:<14}{r["trigger_rate"]:>10.2%}{r["recall"]:>10.1%}'
          f'{r["false_hit_rate"]:>10.2%}{r["precision"]:>10.1%}')

assert res['L0+L1+L2']['recall'] > res['L0 only']['recall'], '加层数召回必然上升'
assert res['L0+L1+L2']['false_hit_rate'] > res['L0 only']['false_hit_rate'], '误伤率也必然上升'
print('\n✅ 加层数：召回上升，**误伤率也上升**——这就是为什么要分层而不是一股脑全上。')
print('   L0 是确定性的（对合法输出永不触发），可以放心**拦截**；')
print(f'   L2 的误伤率 {res["L0+L1+L2"]["false_hit_rate"]:.1%} 落到真实用户身上，')
print('   所以它通常只该**降级或标记**，不该直接拒答。')
print('\n   ⚠️ 这与模块 04 的 CI 门禁完全同构：误伤率过高 → guardrail 被降级成「只记录」→ 被忘掉。')

## 5 · 抽样：尾部采样的选择偏倚

In [ ]:
def head_sampling(rows, rate, seed=0):
    """头部采样：请求开始时就决定记不记 —— 与结果无关，因此无偏。"""
    rng = random.Random(seed)
    return [r for r in rows if rng.random() < rate]

def tail_sampling(rows, base_rate, anomaly_rate, seed=0):
    """尾部采样：出错/异常的全记，正常的按低比例记 —— 信息量高但有强选择偏倚。"""
    rng = random.Random(seed)
    out = []
    for r in rows:
        anomalous = (not r['good']) or r['refused'] or r['reasked']
        p = anomaly_rate if anomalous else base_rate
        if rng.random() < p:
            out.append(dict(r, _sample_kind='anomaly' if anomalous else 'normal',
                            _sample_p=p))
    return out

true_q = float(np.mean([r['good'] for r in sample]))
head = head_sampling(sample, 0.05, seed=1)
tail = tail_sampling(sample, base_rate=0.01, anomaly_rate=1.0, seed=1)

q_head = float(np.mean([r['good'] for r in head]))
q_tail_naive = float(np.mean([r['good'] for r in tail]))
# 逆概率加权（IPW）修正尾部采样的偏倚
w = np.array([1.0 / r['_sample_p'] for r in tail])
q_tail_ipw = float(np.sum(w * np.array([r['good'] for r in tail], float)) / np.sum(w))

print(f'真实质量               {true_q:.1%}')
print(f'头部采样（5%, n={len(head)}）  {q_head:.1%}   偏差 {q_head-true_q:+.1%}')
print(f'尾部采样朴素平均（n={len(tail)}） {q_tail_naive:.1%}   偏差 {q_tail_naive-true_q:+.1%}  ← 严重低估')
print(f'尾部采样 + IPW 加权         {q_tail_ipw:.1%}   偏差 {q_tail_ipw-true_q:+.1%}')
assert abs(q_head - true_q) < 0.02, '头部采样无偏'
assert q_tail_naive < true_q - 0.15, '尾部采样朴素平均严重低估质量'
assert abs(q_tail_ipw - true_q) < abs(q_tail_naive - true_q), 'IPW 必须改善'
print('\n✅ 尾部采样的数据**不能直接算平均**——它按定义就富集了异常样本。')
print('   要么做 IPW 加权，要么只把它当诊断素材而不是统计样本。')
print('   → 实践组合：**头部固定比例（无偏统计）+ 尾部全量记录异常（诊断）**，')
print('     并且用一个字段标明这条数据是怎么被采到的——没有它，两类数据混在一起就再也分不开。')

## 6 · 闭环健康度：回灌延迟、复发率、来源构成

In [ ]:
def loop_health(incidents, task_set, today=100):
    """incidents: [{'id','found_day','ingested_day'(可None),'recurred':bool}]
    task_set: [{'task_id','source'}]"""
    ingested = [i for i in incidents if i['ingested_day'] is not None]
    lat = [i['ingested_day'] - i['found_day'] for i in ingested]
    recur = [i for i in ingested if i['recurred']]
    src = Counter(t['source'] for t in task_set)
    n = max(len(task_set), 1)
    return {
        'n_incidents': len(incidents),
        'ingest_rate': len(ingested) / max(len(incidents), 1),
        'median_latency_days': float(np.median(lat)) if lat else float('nan'),
        'p90_latency_days': float(np.percentile(lat, 90)) if lat else float('nan'),
        'recurrence_rate': len(recur) / max(len(ingested), 1),
        'prod_share': src.get('from_production', 0) / n,
    }

rng = np.random.default_rng(5)
INCIDENTS = []
for i in range(60):
    found = int(rng.integers(0, 80))
    ingested = found + int(rng.gamma(2, 6)) if rng.random() < 0.75 else None
    INCIDENTS.append({'id': f'inc{i:03d}', 'found_day': found,
                      'ingested_day': ingested,
                      'recurred': bool(rng.random() < 0.08)})
TASK_SET = ([{'task_id': f'm{i}', 'source': 'manual'} for i in range(340)]
            + [{'task_id': f'p{i}', 'source': 'from_production'} for i in range(160)])

h = loop_health(INCIDENTS, TASK_SET)
THRESH = {'ingest_rate': ('>=', 0.70), 'median_latency_days': ('<=', 14),
          'p90_latency_days': ('<=', 30), 'recurrence_rate': ('<=', 0.10),
          'prod_share': ('<=', 0.30)}
print(f"{'指标':<24}{'实测':>10}{'阈值':>14}{'状态':>8}")
alerts = []
for k, (op, v) in THRESH.items():
    cur = h[k]
    ok = (cur >= v) if op == '>=' else (cur <= v)
    if not ok:
        alerts.append(k)
    fmt = f'{cur:.1%}' if k in ('ingest_rate', 'recurrence_rate', 'prod_share') else f'{cur:.1f}'
    print(f'{k:<24}{fmt:>10}{f"{op} {v}":>14}{"✅" if ok else "⚠️":>8}')

assert 'prod_share' in alerts, '本例中线上回灌占比超了上限'
print(f'\n未达标: {alerts}')
print(f'\n⚠️ prod_share = {h["prod_share"]:.0%} > 30% —— 任务集正在漂成一个「疑难杂症集」。')
print('   这些 case 按定义是模型的弱项，占比过高会让任务集不再代表真实流量分布。')
print('   → 缓解：给 from_production 设占比上限，并同时从线上做**均匀随机**采样补充「正常」任务。')
print('\n✅ 三个健康指标里，**复发率**是闭环有没有真的起作用的唯一硬指标——')
print('   已经进了任务集的问题又在线上出现，说明回归测试没起到作用。')

## ✏️ 练习 1：漂移告警的分级

实现 `drift_verdict(psi_value, quality_delta=None)`：
- `psi_value < 0.1` → `'stable'`
- `0.1 <= psi < 0.25` → `'watch'`
- `psi >= 0.25` 且 `quality_delta is None` → `'investigate'`（触发带标签抽样）
- `psi >= 0.25` 且 `quality_delta <= -0.03` → `'degraded'`
- `psi >= 0.25` 且 `quality_delta > -0.03` → `'benign_shift'`（变了但没变差）

In [ ]:
def drift_verdict(psi_value, quality_delta=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert drift_verdict(0.05) == 'stable'
assert drift_verdict(0.18) == 'watch'
assert drift_verdict(0.40) == 'investigate'
assert drift_verdict(0.40, -0.08) == 'degraded'
assert drift_verdict(0.40, +0.01) == 'benign_shift'
p_verbose = psi([o['resp_len'] for o in base],
                [o['resp_len'] for o in SCENARIOS['模型变啰嗦']])
print(f'「模型变啰嗦」的 PSI(响应长度) = {p_verbose:.3f}')
print(f'  只看 PSI:            {drift_verdict(p_verbose)}')
print(f'  抽样评测后（质量没变）: {drift_verdict(p_verbose, dq_verbose)}')
assert drift_verdict(p_verbose) == 'investigate'
assert drift_verdict(p_verbose, dq_verbose) == 'benign_shift'
print('✅ 练习 1 通过：`investigate` 这一档是关键——')
print('   它把「触发带标签抽样」和「判定为退化」分成了两步，')
print('   避免了「PSI 高就回滚」这种把「变了」误当成「变差了」的错误。')

## ✏️ 练习 2：Guardrail 的期望代价

实现 `guardrail_cost(recall, fpr, bad_rate, cost_miss=10.0, cost_false_hit=1.0)`：
返回 `{'miss_cost', 'false_hit_cost', 'total'}`，
其中漏放代价 = `bad_rate * (1-recall) * cost_miss`，
误伤代价 = `(1-bad_rate) * fpr * cost_false_hit`。
用它找出「加 L2 到底划不划算」。

In [ ]:
def guardrail_cost(recall, fpr, bad_rate, cost_miss=10.0, cost_false_hit=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
BAD = 1 - true_q
c_l1 = guardrail_cost(res['L0+L1']['recall'], res['L0+L1']['false_hit_rate'], BAD)
c_l2 = guardrail_cost(res['L0+L1+L2']['recall'], res['L0+L1+L2']['false_hit_rate'], BAD)
print(f'坏回答比例 {BAD:.1%}')
print(f"{'配置':<12}{'漏放代价':>12}{'误伤代价':>12}{'总代价':>10}")
for name, c in [('L0+L1', c_l1), ('L0+L1+L2', c_l2)]:
    print(f'{name:<12}{c["miss_cost"]:>12.4f}{c["false_hit_cost"]:>12.4f}{c["total"]:>10.4f}')
assert c_l2['miss_cost'] < c_l1['miss_cost'], '加 L2 减少漏放'
assert c_l2['false_hit_cost'] > c_l1['false_hit_cost'], '但增加误伤'
better = 'L0+L1+L2' if c_l2['total'] < c_l1['total'] else 'L0+L1'
print(f'\n代价比 10:1 时更优的配置: {better}')
# 误伤代价变高时结论会翻转
c_l1b = guardrail_cost(res['L0+L1']['recall'], res['L0+L1']['false_hit_rate'], BAD, 10.0, 8.0)
c_l2b = guardrail_cost(res['L0+L1+L2']['recall'], res['L0+L1+L2']['false_hit_rate'], BAD, 10.0, 8.0)
print(f'误伤代价提到 8 时: L0+L1 {c_l1b["total"]:.4f} vs L0+L1+L2 {c_l2b["total"]:.4f}')
assert c_l2b['total'] < c_l1b['total'], '误伤代价 8 时 L2 仍然更优'
# 继续提高误伤代价，直到结论翻转 —— 找出那个临界点
flip = None
for cf in [1, 2, 4, 8, 12, 16, 20, 30, 50]:
    a = guardrail_cost(res['L0+L1']['recall'], res['L0+L1']['false_hit_rate'], BAD, 10.0, cf)
    b = guardrail_cost(res['L0+L1+L2']['recall'], res['L0+L1+L2']['false_hit_rate'], BAD, 10.0, cf)
    if b['total'] > a['total']:
        flip = cf
        break
print(f'\n误伤代价涨到 {flip} 时（漏放代价固定为 10），结论翻转为 L0+L1 更优')
assert flip is not None and flip > 8
print('✅ 练习 2 通过：「该不该加 L2」没有普遍答案——它取决于漏放与误伤的**代价比**。')
print(f'   本例的临界比是 10 : {flip}，也就是说：')
print('   **只有当「误伤一个好回答」的代价超过「漏放一个坏回答」的一倍多时，L2 才不划算。**')
print('   这个比值必须由业务显式给出，不能由工程师默认——')
print('   而在「拒答」这种直接伤害用户体验的动作上，误伤代价往往比想象中高得多。')

## ✏️ 练习 3：采样方式的标注与合并

实现 `combined_estimate(rows)`：输入带 `_sample_kind` 与 `_sample_p` 字段的样本，
返回 `{'naive', 'ipw', 'n_normal', 'n_anomaly'}`。
`naive` 是朴素平均，`ipw` 是逆概率加权平均。

In [ ]:
def combined_estimate(rows):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ce = combined_estimate(tail)
print(f'真实质量 {true_q:.1%}')
print(f'朴素平均 {ce["naive"]:.1%} (n_normal={ce["n_normal"]}, n_anomaly={ce["n_anomaly"]})')
print(f'IPW 加权 {ce["ipw"]:.1%}')
assert ce['n_anomaly'] > 0 and ce['n_normal'] > 0
assert abs(ce['ipw'] - true_q) < abs(ce['naive'] - true_q)
# 采样率相同时两者应当一致
uniform = [dict(r, _sample_kind='normal', _sample_p=0.05) for r in head]
cu = combined_estimate(uniform)
assert abs(cu['naive'] - cu['ipw']) < 1e-9, '均匀采样时 IPW 退化为朴素平均'
print('\n均匀采样时: 朴素 = IPW ✓（IPW 在无偏采样下自动退化成不做任何事）')
print('✅ 练习 3 通过：`_sample_p` 这个字段是把两类数据合并统计的唯一途径——')
print('   不记它，尾部采样的数据就只能当诊断素材，永远进不了统计。')

## ✏️ 练习 4：闭环的复发率追踪

实现 `recurrence_report(incidents, window_days=60)`：
只统计「入库后至少观察了 window_days」的事件，返回
`{'n_tracked', 'n_recurred', 'recurrence_rate', 'healthy'}`，
`healthy` 表示复发率 ≤ 10%。假设今天是第 100 天。

In [ ]:
def recurrence_report(incidents, window_days=60, today=100):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rr = recurrence_report(INCIDENTS)
print(f'追踪中的事件 {rr["n_tracked"]} 条 | 复发 {rr["n_recurred"]} 条 | '
      f'复发率 {rr["recurrence_rate"]:.1%} | 健康={rr["healthy"]}')
assert rr['n_tracked'] < len([i for i in INCIDENTS if i['ingested_day'] is not None]), \
    '未观察满窗口的事件应被排除'
assert 0 <= rr['recurrence_rate'] <= 1
bad = [{'id': 'x', 'found_day': 0, 'ingested_day': 1, 'recurred': True}] * 10
assert recurrence_report(bad)['healthy'] is False
empty = recurrence_report([{'id': 'y', 'found_day': 90, 'ingested_day': 95, 'recurred': False}])
assert empty['n_tracked'] == 0 and math.isnan(empty['recurrence_rate'])
print('\n刚入库不久的事件被正确排除（观察窗口不足），空样本返回 nan ✓')
print('✅ 练习 4 通过：复发率是闭环有没有真的起作用的**唯一硬指标**——')
print('   回灌延迟再短，如果问题照样复发，说明那道题写得不对（判分器没抓住真正的失败模式）。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def drift_verdict(psi_value, quality_delta=None):
    if psi_value < 0.1:
        return 'stable'
    if psi_value < 0.25:
        return 'watch'
    if quality_delta is None:
        return 'investigate'
    return 'degraded' if quality_delta <= -0.03 else 'benign_shift'

In [ ]:
# 练习 2 参考答案
def guardrail_cost(recall, fpr, bad_rate, cost_miss=10.0, cost_false_hit=1.0):
    miss = bad_rate * (1 - recall) * cost_miss
    fh = (1 - bad_rate) * fpr * cost_false_hit
    return {'miss_cost': miss, 'false_hit_cost': fh, 'total': miss + fh}

In [ ]:
# 练习 3 参考答案
def combined_estimate(rows):
    if not rows:
        return {'naive': float('nan'), 'ipw': float('nan'),
                'n_normal': 0, 'n_anomaly': 0}
    vals = np.array([float(r['good']) for r in rows])
    w = np.array([1.0 / r['_sample_p'] for r in rows])
    return {'naive': float(vals.mean()),
            'ipw': float((w * vals).sum() / w.sum()),
            'n_normal': sum(1 for r in rows if r['_sample_kind'] == 'normal'),
            'n_anomaly': sum(1 for r in rows if r['_sample_kind'] == 'anomaly')}

In [ ]:
# 练习 4 参考答案
def recurrence_report(incidents, window_days=60, today=100):
    tracked = [i for i in incidents
               if i['ingested_day'] is not None
               and today - i['ingested_day'] >= window_days]
    n = len(tracked)
    rec = sum(1 for i in tracked if i['recurred'])
    rate = (rec / n) if n else float('nan')
    return {'n_tracked': n, 'n_recurred': rec, 'recurrence_rate': rate,
            'healthy': bool(n and rate <= 0.10)}

---
## 🧪 真实工程胶囊：线上监控的落地

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 埋点：沿用 OpenTelemetry GenAI 语义约定（C66-03 已经讲过一次）
# ══════════════════════════════════════════════════════════════════
with tracer.start_as_current_span("gen_ai.chat") as sp:
    sp.set_attribute("gen_ai.request.model", MODEL_ID)
    sp.set_attribute("gen_ai.usage.input_tokens", n_in)
    sp.set_attribute("gen_ai.usage.output_tokens", n_out)
    # 监控必需的自定义属性:
    sp.set_attribute("app.intent", intent)              # 分层与漂移
    sp.set_attribute("app.guardrail.layer", hit_layer)  # guardrail 触发在哪层
    sp.set_attribute("app.sample_kind", kind)           # head | tail_anomaly ← **必须记**
    sp.set_attribute("app.sample_p", p)                 # 逆概率加权需要
# 没有 sample_kind / sample_p，头部与尾部采样的数据混在一起就再也分不开了。

# ══════════════════════════════════════════════════════════════════
# B. 每日自动跑的监控（全部无标签，成本近乎为零）
# ══════════════════════════════════════════════════════════════════
DAILY = {
  "psi_request_length":  lambda: psi(baseline.req_len, today.req_len),
  "psi_intent":          lambda: psi_categorical(baseline.intent, today.intent),
  "resp_len_p50":        ...,   # 行为漂移（C67-05 的 hack 形态）
  "md_density":          ...,
  "refusal_rate":        ...,
  "reask_rate":          ...,   # 用户没被满足（最有价值的无标签信号）
  "self_consistency":    ...,   # 抽 1% 请求跑两次
  "guardrail_trigger":   ...,   # 按层分开
}
# 告警规则用 drift_verdict（练习 1）：
#   PSI 高 → **investigate**（触发带标签抽样），而不是直接回滚。

# ══════════════════════════════════════════════════════════════════
# C. 每周人工抽样（三段结构，与 C67-03 的元评测集同构）
# ══════════════════════════════════════════════════════════════════
# main_sample/      均匀随机 200 条  → 无偏估计整体质量
# uncertainty/      低置信/swap 不一致 100 条 → 判分器与模型的边界
# incident/         guardrail 触发 + 负反馈 全量 → 该修什么
# **三者分开统计，绝不混算。**

# ══════════════════════════════════════════════════════════════════
# D. 闭环的四个细节（讲解第 6 节）
# ══════════════════════════════════════════════════════════════════
# 1. 去标识化在**采样时**做，不是入库后
# 2. 人工确认「这确实是模型的问题」——guardrail 触发 != 模型错了
# 3. 写成可判分的任务（C66-01 的六条规则），不是直接贴原文
# 4. 进任务集**新版本**（C68-01 只增不改），source='from_production'
#    并设占比上限（<= 30%），否则任务集会漂成「疑难杂症集」
#
# 健康指标: 回灌延迟中位 <= 14 天 · 复发率 <= 10% · prod_share <= 30%
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 两层不匹配 | 分布 vs 口径；先查覆盖率，再谈效应量 | 「离线涨了线上没涨」 |
| 重加权 | 用线上分布给离线分数加权，才是可比的口径 | 离线-在线对齐 |
| PSI/KS 的局限 | 能告诉你「变了」，不能告诉你「变差了」 | 漂移告警的动作 |
| 三个无标签信号 | 自一致性 / 行为分布 / 重问率，覆盖三个失败方向 | 每日监控 |
| Guardrail 分层 | L0 可拦截，L2 通常只该降级——误伤落在真实用户身上 | 实时护栏 |
| 尾部采样的偏倚 | 不能直接算平均；必须记 `sample_kind` 与 `sample_p` | 采样设计 |
| 闭环健康度 | 复发率是唯一硬指标；`from_production` 要设占比上限 | 反馈回路 |

**全课收尾**：01 声明 → 02 执行 → 03 存储 → 04 门禁 → 05 线上与回灌。
五层服务同一个目标：**让「这个数字变了」这件事有唯一的解释。**

**下一门课 C69 · Agent 安全与提示注入**：本课的 guardrail 是质量护栏，
而当输入里有人**主动**想让 agent 做坏事时，需要的是另一套东西——
攻击面在哪、怎么量它、怎么防。